# Comparative Emotion Classification in Short User-Generated Text

This notebook implements the CM3060 NLP mid-term coursework project.

The project compares a traditional statistical text classifier with an embedding-based neural model for emotion classification in short user-generated text. It uses Google Research's GoEmotions dataset and filters the original labels to six target classes: `joy`, `anger`, `fear`, `sadness`, `surprise`, and `neutral`.

## 1. Setup

Run the install cell if the required libraries are missing. If your environment already has these packages, you can skip it.

In [ ]:
# Optional install cell. Uncomment and run if needed.
# %pip install -q pandas numpy scikit-learn matplotlib seaborn datasets tensorflow

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Embedding, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

sns.set_theme(style="whitegrid")

## 2. Load and Filter GoEmotions

GoEmotions contains 27 fine-grained emotion categories plus neutral. To keep the coursework manageable, this project filters the dataset to six existing labels only. The comments are not manually relabelled.

In [ ]:
raw_dataset = load_dataset("go_emotions", "simplified")
raw_dataset

In [ ]:
label_names = raw_dataset["train"].features["labels"].feature.names
target_label_names = ["joy", "anger", "fear", "sadness", "surprise", "neutral"]
target_label_ids = [label_names.index(label) for label in target_label_names]
target_id_to_new_id = {old_id: new_id for new_id, old_id in enumerate(target_label_ids)}

print("Original label count:", len(label_names))
print("Target labels:", target_label_names)
print("Target label IDs:", target_label_ids)

In [ ]:
def keep_single_target_label(example):
    labels = example["labels"]
    return len(labels) == 1 and labels[0] in target_label_ids


def convert_to_six_class_label(example):
    old_label_id = example["labels"][0]
    new_label_id = target_id_to_new_id[old_label_id]
    return {
        "text": example["text"],
        "label": new_label_id,
        "label_name": target_label_names[new_label_id],
    }


dataset_filtered = raw_dataset.filter(keep_single_target_label)
dataset_filtered = dataset_filtered.map(
    convert_to_six_class_label,
    remove_columns=raw_dataset["train"].column_names,
)

dataset_filtered

In [ ]:
train_df = dataset_filtered["train"].to_pandas()
val_df = dataset_filtered["validation"].to_pandas()
test_df = dataset_filtered["test"].to_pandas()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

train_df.head()

## 3. Exploratory Data Analysis

The class distribution matters because imbalanced classes can make accuracy misleading. Per-class F1-score and confusion matrices are therefore included later.

In [ ]:
class_counts = train_df["label_name"].value_counts().reindex(target_label_names)
display(class_counts)

plt.figure(figsize=(8, 4))
sns.barplot(x=class_counts.index, y=class_counts.values)
plt.title("Training Set Class Distribution")
plt.xlabel("Emotion")
plt.ylabel("Number of comments")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
for label in target_label_names:
    sample = train_df[train_df["label_name"] == label].sample(1, random_state=RANDOM_STATE)
    print(f"\n[{label}]")
    print(sample["text"].iloc[0])

## 4. Preprocessing

For the statistical models, preprocessing is handled inside scikit-learn vectorizers. TF-IDF and bag-of-words represent text using token frequency patterns.

For the embedding model, text is tokenized into integer sequences and padded to a fixed length so that it can be passed into a neural network embedding layer.

In [ ]:
X_train = train_df["text"].astype(str)
y_train = train_df["label"].astype(int)

X_val = val_df["text"].astype(str)
y_val = val_df["label"].astype(int)

X_test = test_df["text"].astype(str)
y_test = test_df["label"].astype(int)

num_classes = len(target_label_names)

## 5. Evaluation Helper

In [ ]:
results = []


def evaluate_model(model_name, y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    weighted_f1 = f1_score(y_true, y_pred, average="weighted")

    results.append({
        "model": model_name,
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    })

    print(f"{model_name}")
    print("-" * len(model_name))
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}\n")
    print(classification_report(y_true, y_pred, target_names=target_label_names))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(7, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=target_label_names,
        yticklabels=target_label_names,
    )
    plt.title(f"Confusion Matrix: {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

## 6. Baseline Model

The baseline predicts the most frequent class in the training data. This gives a minimum benchmark that useful models should beat.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

evaluate_model("Majority-Class Baseline", y_test, baseline_pred)

## 7. Statistical Models

The first statistical model uses bag-of-words with Multinomial Naive Bayes, matching the course topic on Bayesian text classification. The second uses TF-IDF with Logistic Regression as a stronger traditional baseline.

In [ ]:
nb_model = Pipeline([
    ("vectorizer", CountVectorizer(lowercase=True, stop_words="english", min_df=2)),
    ("classifier", MultinomialNB()),
])

nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)

evaluate_model("Bag-of-Words + Multinomial Naive Bayes", y_test, nb_pred)

In [ ]:
tfidf_lr_model = Pipeline([
    ("vectorizer", TfidfVectorizer(lowercase=True, stop_words="english", min_df=2, ngram_range=(1, 2))),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
])

tfidf_lr_model.fit(X_train, y_train)
tfidf_lr_pred = tfidf_lr_model.predict(X_test)

evaluate_model("TF-IDF + Logistic Regression", y_test, tfidf_lr_pred)

## 8. Embedding-Based Neural Model

This model learns a task-specific embedding for words in the filtered GoEmotions data. The architecture is intentionally simple so that the comparison focuses on representation and classification approach rather than excessive model complexity.

In [ ]:
max_words = 12000
max_len = 40
embedding_dim = 64

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_len, padding="post", truncating="post")
X_val_seq = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=max_len, padding="post", truncating="post")
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_len, padding="post", truncating="post")

y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
y_val_cat = tf.keras.utils.to_categorical(y_val, num_classes=num_classes)

In [ ]:
embedding_model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
    GlobalAveragePooling1D(),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(num_classes, activation="softmax"),
])

embedding_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

embedding_model.summary()

In [ ]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
)

history = embedding_model.fit(
    X_train_seq,
    y_train_cat,
    validation_data=(X_val_seq, y_val_cat),
    epochs=10,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1,
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("Embedding Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Embedding Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
embedding_probs = embedding_model.predict(X_test_seq)
embedding_pred = embedding_probs.argmax(axis=1)

evaluate_model("Keras Embedding Neural Network", y_test, embedding_pred)

## 9. Model Comparison

In [ ]:
results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
display(results_df)

plt.figure(figsize=(9, 4))
sns.barplot(data=results_df, x="model", y="macro_f1")
plt.title("Model Comparison by Macro F1-Score")
plt.xlabel("Model")
plt.ylabel("Macro F1")
plt.xticks(rotation=30, ha="right")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## 10. Error Analysis

Reviewing misclassified examples helps explain why some emotions are harder to identify. This is useful for the coursework discussion section.

In [ ]:
analysis_df = test_df.copy()
analysis_df["predicted_label"] = [target_label_names[i] for i in tfidf_lr_pred]
analysis_df["true_label"] = [target_label_names[i] for i in y_test]
errors = analysis_df[analysis_df["true_label"] != analysis_df["predicted_label"]]

print("Number of TF-IDF Logistic Regression errors:", len(errors))
errors[["text", "true_label", "predicted_label"]].sample(
    min(10, len(errors)),
    random_state=RANDOM_STATE,
)

## 11. Notes for the Written Report

Use the results above to write the report sections required by the brief:

- Domain area: emotion detection in short user-generated online communication.
- Objective: compare statistical and embedding-based models for multi-class emotion classification.
- Dataset: Google Research GoEmotions, filtered to six target labels.
- Preprocessing: target-label filtering, train/validation/test splits, vectorization for statistical models, tokenization and padding for embedding models.
- Baseline: majority-class classifier.
- Comparative models: bag-of-words Naive Bayes, TF-IDF Logistic Regression, and Keras embedding neural network.
- Evaluation: accuracy, macro F1, weighted F1, classification report, confusion matrices, and error examples.